# MSFT Next-Day Direction Predictor — Iteration 5: Moirai

**Author:** Lucan den Dekker  
**Project:** ADSAI Group 15 Block D Capstone  
**Date:** 2026-06-03  
**Pre-trained model:** Moirai (Salesforce, 2024) — Universal Foundation Model for Time Series

---

**Improvements over Iteration 4**

| Change | Reason |
|---|---|
| Moirai replaces ARIMAX | Foundation model trained on billions of diverse time series — zero-shot, no fitting required |
| Multivariate context natively | MSFT close + SPY + VIX + gold + oil fed simultaneously as context |
| Probabilistic forecasting | 100 forecast samples → median for direction, quantiles for confidence |
| No model training required | Moirai generalises directly from pre-trained weights — no overfitting risk |
| Uncertainty intervals | `yhat_lower` / `yhat_upper` from sample quantiles — useful for dashboard confidence display |

| Section | Content |
|---|---|
| 0 | Imports and Setup |
| 1 | Load Data |
| 2 | Feature Engineering (for target labels) |
| 3 | Train/Test Split + Multivariate Context |
| 4 | Baseline Model |
| 5 | Moirai: Zero-Shot Walk-Forward Forecast + Direction Signal |
| 5b | Threshold Sweep: Precision vs Coverage |
| 6 | Model Comparison (Moirai vs Baseline) |
| 7 | Error Analysis |
| 8 | Business Value Interpretation |

In [3]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args, "-q"])

# These have no numpy conflicts
pip("einops", "huggingface_hub", "jaxtyping")

# These have numpy<2 constraints → install without dep-checks
pip("gluonts", "hydra-core", "--no-deps")

# uni2ts from GitHub, skip deps too
pip("git+https://github.com/SalesforceAIResearch/uni2ts.git", "--no-deps")

print(f"Python : {sys.executable}")
print("Done — restart kernel, then run from Section 0.")

Python : c:\Users\lucan\anaconda3\envs\myenv\python.exe
Done — restart kernel, then run from Section 0.


---
## Section 0 — Imports and Setup

**Moirai** (Salesforce, 2024) is a Universal Time Series Forecasting Transformer:
- Pre-trained on the LOTSA dataset: 27 billion observations from diverse domains
- Supports univariate and multivariate input
- Probabilistic: returns a distribution of forecasts (not a single point)
- Zero-shot: no fine-tuning needed — the pre-trained weights generalise directly

Model size: `moirai-1.0-R-small` (~14M parameters, fastest).
Alternatives: `moirai-1.0-R-base` (~91M) or `moirai-1.0-R-large` (~311M).

`CONTEXT_LENGTH = 128` means Moirai sees the last 128 trading days (~6 months) to
forecast tomorrow's price. `NUM_SAMPLES = 100` draws 100 forecast paths from the
probabilistic distribution; we use the median as our point estimate.

In [ ]:
import sys
import types
from importlib.machinery import ModuleSpec

# ── Packages to stub ──────────────────────────────────────────────────────────
_MOCK_PREFIXES = ("gluonts", "lightning", "pydantic", "pydantic_core",
                  "hydra", "omegaconf", "antlr4")

# Purge stale copies (including partial uni2ts.model.moirai.* from previous runs)
for _k in list(sys.modules.keys()):
    if (any(_k == p or _k.startswith(p + ".") for p in _MOCK_PREFIXES)
            or _k.startswith("uni2ts.model.moirai")):
        del sys.modules[_k]


class _AutoMock:
    """Python 3.4+ meta-path finder/loader — blank package for stubbed prefixes.

    Uses find_spec / create_module / exec_module (the current protocol).
    find_module / load_module were removed in Python 3.12+.
    """

    def find_spec(self, fullname, path, target=None):
        if any(fullname == p or fullname.startswith(p + ".")
               for p in _MOCK_PREFIXES):
            return ModuleSpec(fullname, self, is_package=True)
        return None

    def create_module(self, spec):
        m = types.ModuleType(spec.name)
        m.__name__    = spec.name
        m.__path__    = []   # marks module as a package → submodule imports work
        m.__package__ = spec.name.rsplit(".", 1)[0] if "." in spec.name else spec.name
        m.__loader__  = self
        m.__spec__    = spec
        if spec.name == "pydantic":
            m.__version__ = "2.0.0"
        sys.modules[spec.name] = m
        return m

    def exec_module(self, module):
        pass  # nothing to execute — blank module


# Register FIRST in meta_path so it runs before all real finders
sys.meta_path = [f for f in sys.meta_path if not isinstance(f, _AutoMock)]
sys.meta_path.insert(0, _AutoMock())

# Also stub finetune + pretrain entirely so those files never execute
def _stub(name, **attrs):
    m = types.ModuleType(name)
    m.__path__ = []
    for k, v in attrs.items():
        setattr(m, k, v)
    sys.modules[name] = m

_stub("uni2ts.model.moirai.finetune",
      MoiraiFinetune=type("MoiraiFinetune", (), {}))
_stub("uni2ts.model.moirai.pretrain",
      MoiraiPretrain=type("MoiraiPretrain", (), {}))

# ── Standard imports ──────────────────────────────────────────────────────────
import warnings
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)
import torch
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
import joblib

# ── Project setup ─────────────────────────────────────────────────────────────
RANDOM_SEED: int = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("msft_modelling")

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    plt.style.use("seaborn-darkgrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13

NEUTRAL_THRESHOLD: float = 0.2
CONTEXT_LENGTH:   int   = 128
NUM_SAMPLES:      int   = 100
MOIRAI_SIZE:      str   = "small"
PATCH_SIZE:       int   = 16


def find_project_root(marker: str = "README.md") -> Path:
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    return current


PROJECT_ROOT = find_project_root()
DATA_PATH    = PROJECT_ROOT / "data" / "processed" / "msft_merged.csv"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "msft-forecaster" / "data" / "processed" / "msft_merged.csv"

MODELS_DIR = PROJECT_ROOT / "models" / "lucan"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logger.info("Device       : %s", DEVICE)
logger.info("Project root : %s", PROJECT_ROOT)
logger.info("Imports OK   : %s / %s", MoiraiForecast.__name__, MoiraiModule.__name__)

---
## Section 1 — Load Data

Full 2015–2026 dataset. Moirai receives the raw `Close` price as its target series
and SPY / VIX / gold / oil as covariate series. All five series are already in
`msft_merged.csv`.

In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    """Load the MSFT merged CSV and return a date-indexed DataFrame."""
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")
    df = pd.read_csv(path)
    date_col = next(
        (c for c in df.columns if c.lower() in ("date", "datetime", "timestamp")), None
    )
    if date_col is None:
        raise ValueError(f"No date column found. Columns: {list(df.columns)}")
    df[date_col] = pd.to_datetime(df[date_col])
    df.set_index(date_col, inplace=True)
    df.index.name = "Date"
    df.sort_index(inplace=True)
    return df


df_raw = load_data(DATA_PATH)
logger.info("Loaded: shape=%s | %s to %s",
            df_raw.shape, df_raw.index.min().date(), df_raw.index.max().date())

if df_raw.isnull().sum().sum() > 0:
    logger.warning("Missing values found; dropping.")
    df_raw = df_raw.dropna()
else:
    logger.info("No missing values.")

print(df_raw.head())

---
## Section 2 — Feature Engineering (Target Labels)

Same as it4: binary UP/DOWN labels for evaluation. Moirai does not use these
for training — only for measuring direction prediction accuracy.

In [ ]:
def _detect_column(df: pd.DataFrame, candidates: list) -> str:
    """Return the first column name matching any candidate (case-insensitive)."""
    col_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name in col_lower:
            return col_lower[name]
    raise KeyError(f"None of {candidates} found in columns: {list(df.columns)}")


def engineer_features(df: pd.DataFrame, threshold: float = NEUTRAL_THRESHOLD) -> pd.DataFrame:
    """Compute binary direction target labels from MSFT close prices."""
    df = df.copy()
    close = df[_detect_column(df, ["close", "msft_close", "adj close", "adj_close"])]
    next_day_return = (close.shift(-1) - close) / close * 100
    df["target"] = np.select(
        [next_day_return > threshold, next_day_return < -threshold], [1, 0], default=np.nan,
    )
    df = df.dropna(subset=["target"])
    df["target"] = df["target"].astype(int)
    df = df.dropna()
    class_counts = df["target"].value_counts().sort_index()
    logger.info("Threshold=%.1f%%  DOWN(0): %d | UP(1): %d | total: %d",
                threshold, class_counts.get(0, 0), class_counts.get(1, 0), len(df))
    return df


df = engineer_features(df_raw)
close_col = _detect_column(df_raw, ["close", "msft_close", "adj close", "adj_close"])
spy_col   = _detect_column(df_raw, ["spy_close", "spy"])
vix_col   = _detect_column(df_raw, ["vix"])
gold_col  = _detect_column(df_raw, ["gold_close", "gold"])
oil_col   = _detect_column(df_raw, ["oil_close", "oil"])

---
## Section 3 — Train/Test Split + Multivariate Context

The 80/20 feature split provides target labels. For Moirai's context window:
- **Target series**: MSFT close (normalised to returns inside the model)
- **Covariates**: SPY close, VIX, gold close, oil close — provided as-is
  (Moirai normalises internally)

The full historical series is kept so Moirai can slide a `CONTEXT_LENGTH=128` day
window across the entire test period. No separate training window is needed — the
model is zero-shot.

In [ ]:
# ── Feature split (for target labels) ────────────────────────────────────────
split_idx = int(len(df) * 0.80)
df_train  = df.iloc[:split_idx]
df_test   = df.iloc[split_idx:]

logger.info("Feature test : %d rows | %s to %s",
            len(df_test), df_test.index.min().date(), df_test.index.max().date())

y_test          = df_test["target"]
test_start_date = df_test.index.min()

# ── Full series for sliding context window ────────────────────────────────────
close_full = df_raw[close_col].dropna()
spy_full   = df_raw[spy_col].dropna()
vix_full   = df_raw[vix_col].dropna()
gold_full  = df_raw[gold_col].dropna()
oil_full   = df_raw[oil_col].dropna()

# Align all series on common trading days
all_series = pd.concat([close_full, spy_full, vix_full, gold_full, oil_full], axis=1)
all_series.columns = ["close", "spy", "vix", "gold", "oil"]
all_series = all_series.dropna()

close_test = all_series.loc[all_series.index >= test_start_date, "close"]

logger.info("Full aligned series : %d rows | %s to %s",
            len(all_series), all_series.index.min().date(), all_series.index.max().date())
logger.info("Test window         : %d rows | %s to %s",
            len(close_test), close_test.index.min().date(), close_test.index.max().date())
logger.info("Context length      : %d trading days", CONTEXT_LENGTH)

print(f"Feature test  : {len(df_test)} rows")
print(f"Test close    : {len(close_test)} rows")
print(f"Covariates    : SPY, VIX, gold, oil")

---
## Section 4 — Baseline Model

Same majority-class baseline as it1–4.

In [ ]:
def evaluate_model(y_true, y_pred, model_name: str) -> dict:
    """Compute weighted classification metrics and return as a results dict."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    logger.info("%s  Acc=%.4f  Prec=%.4f  Rec=%.4f  F1=%.4f",
                model_name, acc, prec, rec, f1)
    return {"Model": model_name, "Accuracy": round(acc, 4),
            "Precision": round(prec, 4), "Recall": round(rec, 4), "F1": round(f1, 4)}


def plot_confusion_matrix(y_true, y_pred, model_name: str) -> None:
    """Plot a seaborn heatmap confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["DOWN (0)", "UP (1)"],
                yticklabels=["DOWN (0)", "UP (1)"], ax=ax)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    plt.show()


dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
dummy.fit(np.zeros((len(y_test), 1)), y_test)
y_pred_dummy = dummy.predict(np.zeros((len(y_test), 1)))

baseline_metrics = evaluate_model(y_test, y_pred_dummy, "Baseline (Majority Class)")
print(classification_report(y_test, y_pred_dummy, target_names=["DOWN", "UP"]))

---
## Section 5 — Moirai: Zero-Shot Walk-Forward Forecast + Direction Signal

### How Moirai forecasts

Moirai uses a **patch-based transformer** architecture:
1. The context window (128 days of prices + covariates) is split into patches of size 16
2. Each patch is embedded via a learned linear projection
3. The transformer attends over all patches to capture long-range dependencies
4. The decoder outputs a **distribution** (100 samples) over the next-day price

The **median** of those 100 samples is used as the point forecast for direction derivation.
The **10th and 90th percentiles** form the uncertainty interval.

### Walk-Forward Protocol

For each test day `d`:
1. Take the last `CONTEXT_LENGTH` days from `all_series` up to and including `d`
2. Pass MSFT close as `past_target` and SPY/VIX/gold/oil as `past_feat_dynamic_real`
3. Pass today's covariate values as `feat_dynamic_real` (the forecast-period covariates)
4. Moirai returns 100 forecast samples for day `d+1`
5. Median sample → direction signal

No model training or updating — the same pre-trained weights are used for every
test step. This is true zero-shot generalisation.

> **Note:** The first run downloads ~150 MB of model weights from HuggingFace.
> Subsequent runs use the local cache. Expect ~0.5–2 minutes for the walk-forward loop.

In [ ]:
# ── Load pre-trained Moirai model ─────────────────────────────────────────────
logger.info("Loading Moirai moirai-1.0-R-%s from HuggingFace ...", MOIRAI_SIZE)

module = MoiraiModule.from_pretrained(f"Salesforce/moirai-1.0-R-{MOIRAI_SIZE}")

moirai_model = MoiraiForecast(
    module=module,
    prediction_length=1,           # one-step-ahead
    context_length=CONTEXT_LENGTH,
    patch_size=PATCH_SIZE,
    num_samples=NUM_SAMPLES,
    target_dim=1,                  # MSFT close (univariate target)
    feat_dynamic_real_dim=4,       # SPY, VIX, gold, oil at forecast time
    past_feat_dynamic_real_dim=4,  # SPY, VIX, gold, oil in context window
).to(DEVICE)
moirai_model.eval()
logger.info("Moirai loaded. Parameters: %s",
            sum(p.numel() for p in moirai_model.parameters()))

# ── Walk-forward forecast ─────────────────────────────────────────────────────
close_test_values = close_test.values.tolist()
forecasts_raw:   list = []   # median forecast
forecasts_q10:   list = []   # 10th percentile
forecasts_q90:   list = []   # 90th percentile

COV_COLS = ["spy", "vix", "gold", "oil"]
N_COV    = len(COV_COLS)

logger.info("Walk-forward over %d test observations ...", len(close_test_values))

for i, _ in enumerate(close_test_values):
    date     = close_test.index[i]
    ctx_end  = date  # context ends on the current date (inclusive)

    # Context window: last CONTEXT_LENGTH rows up to and including today
    ctx_df = all_series.loc[all_series.index <= ctx_end].iloc[-CONTEXT_LENGTH:]

    if len(ctx_df) < CONTEXT_LENGTH:
        # Not enough history yet — fall back to previous forecast or skip
        forecasts_raw.append(float("nan"))
        forecasts_q10.append(float("nan"))
        forecasts_q90.append(float("nan"))
        continue

    # past_target: (1, context_length, 1) — MSFT close
    past_target = torch.tensor(
        ctx_df["close"].values, dtype=torch.float32
    ).unsqueeze(0).unsqueeze(-1).to(DEVICE)

    # past_feat_dynamic_real: (1, context_length, 4) — covariates in context
    past_cov = torch.tensor(
        ctx_df[COV_COLS].values, dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    # feat_dynamic_real: (1, 1, 4) — covariate values at the forecast time step
    # Use today's actual covariate values (known at end-of-day, no lookahead)
    if date in all_series.index:
        future_cov_vals = all_series.loc[date, COV_COLS].values
    else:
        future_cov_vals = ctx_df[COV_COLS].iloc[-1].values  # fallback: last known
    future_cov = torch.tensor(
        future_cov_vals, dtype=torch.float32
    ).unsqueeze(0).unsqueeze(0).to(DEVICE)

    past_observed = torch.ones_like(past_target, dtype=torch.bool)
    past_is_pad   = torch.zeros(1, CONTEXT_LENGTH, dtype=torch.bool).to(DEVICE)

    with torch.no_grad():
        forecast = moirai_model(
            past_target=past_target,
            past_observed_target=past_observed,
            past_is_pad=past_is_pad,
            feat_dynamic_real=future_cov,
            past_feat_dynamic_real=past_cov,
        )

    # forecast.samples: (num_samples, 1, 1)
    samples = forecast.samples.squeeze().cpu().numpy()  # shape: (num_samples,)
    forecasts_raw.append(float(np.median(samples)))
    forecasts_q10.append(float(np.percentile(samples, 10)))
    forecasts_q90.append(float(np.percentile(samples, 90)))

logger.info("Walk-forward complete. %d forecasts generated.", len(forecasts_raw))

# ── Save model ────────────────────────────────────────────────────────────────
moirai_path = MODELS_DIR / "moirai_lucan_it5.pkl"
joblib.dump(moirai_model, moirai_path)
logger.info("Moirai model saved to %s", moirai_path)

# ── Convert forecasts to UP/DOWN direction ─────────────────────────────────────
forecasts_arr  = np.array(forecasts_raw)
current_closes = close_test.values

predicted_returns = np.where(
    np.isnan(forecasts_arr), np.nan,
    (forecasts_arr - current_closes) / current_closes * 100,
)

direction_raw = np.select(
    [predicted_returns > NEUTRAL_THRESHOLD, predicted_returns < -NEUTRAL_THRESHOLD],
    [1, 0],
    default=np.nan,
)
direction_series = pd.Series(direction_raw, index=close_test.index, name="moirai_direction")

common_dates   = df_test.index.intersection(direction_series.index)
y_test_moirai  = df_test.loc[common_dates, "target"]
y_pred_moirai  = direction_series.loc[common_dates]

valid_mask   = y_pred_moirai.notna()
neutral_days = int((~valid_mask).sum())
y_test_moirai = y_test_moirai[valid_mask]
y_pred_moirai = y_pred_moirai[valid_mask].astype(int)

logger.info("Moirai direction: %d predictions, %d neutral/skipped",
            len(y_pred_moirai), neutral_days)

moirai_metrics = evaluate_model(y_test_moirai, y_pred_moirai, "Moirai Direction Signal")
print(classification_report(y_test_moirai, y_pred_moirai, target_names=["DOWN", "UP"]))
plot_confusion_matrix(y_test_moirai, y_pred_moirai, "Moirai Direction Signal")

# ── Forecast vs actual price with uncertainty ─────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(close_test.index, close_test.values,
        color="royalblue", linewidth=1.2, label="Actual Close")
ax.plot(close_test.index, forecasts_arr,
        color="darkorange", linewidth=1.0, linestyle="--", label="Moirai Median Forecast")
ax.fill_between(
    close_test.index,
    np.array(forecasts_q10),
    np.array(forecasts_q90),
    alpha=0.15, color="darkorange", label="10th–90th percentile",
)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate(rotation=30)
ax.set_xlabel("Date")
ax.set_ylabel("Close Price (USD)")
ax.set_title("MSFT Moirai Zero-Shot Forecast vs Actual — Test Window")
ax.legend()
plt.tight_layout()
plt.show()

### Moirai Forecast Interpretation

**Uncertainty intervals** are a key advantage over ARIMA/ARIMAX:
- Wide intervals (large gap between 10th and 90th percentile) = high uncertainty
- Narrow intervals = model is more confident about the next-day price level
- Days where the entire distribution is above today's close → strong UP signal
- Days where the distribution straddles today's price → NEUTRAL (model unsure)

**Why Moirai may outperform ARIMAX:**
- Pre-trained on 27B observations from diverse domains — pattern generalisation
- Multivariate context captures cross-series dependencies (SPY ↔ MSFT) non-linearly
- No parameter estimation on a small dataset — zero overfitting risk

**Why Moirai may not reach 65%:**
- Stock prices are near-random-walk at the 1-day horizon regardless of model complexity
- MSFT is highly efficient — most public information is already priced in

---
## Section 5b — Threshold Sweep: Precision vs Coverage Trade-off

Same as it4: sweep threshold values and show the precision/coverage tradeoff.
Moirai's probabilistic output means we can also use the uncertainty interval
width as an additional confidence filter (future work).

In [ ]:
THRESHOLDS = [0.2, 0.3, 0.5, 0.7, 1.0]
sweep_results = []

for thr in THRESHOLDS:
    dir_raw  = np.where(np.isnan(predicted_returns), np.nan,
                        np.select([predicted_returns > thr, predicted_returns < -thr],
                                  [1, 0], default=np.nan))
    dir_s    = pd.Series(dir_raw, index=close_test.index)
    common   = df_test.index.intersection(dir_s.index)
    y_true_s = df_test.loc[common, "target"]
    y_pred_s = dir_s.loc[common]
    valid_s  = y_pred_s.notna()
    n_signal = int(valid_s.sum())
    coverage = n_signal / len(y_true_s) * 100

    if n_signal == 0:
        sweep_results.append({"Threshold (±%)": thr, "Coverage (% days)": 0.0,
                               "Signal days": 0, "Accuracy": float("nan"), "F1": float("nan")})
        continue

    y_t = y_true_s[valid_s].values
    y_p = y_pred_s[valid_s].astype(int).values
    sweep_results.append({
        "Threshold (±%)": thr,
        "Coverage (% days)": round(coverage, 1),
        "Signal days": n_signal,
        "Accuracy": round(accuracy_score(y_t, y_p), 4),
        "F1": round(f1_score(y_t, y_p, average="weighted", zero_division=0), 4),
    })

sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

best_thr_row   = sweep_df.loc[sweep_df["F1"].idxmax()]
BEST_THRESHOLD = best_thr_row["Threshold (±%)"]
print(f"\nBest threshold: ±{BEST_THRESHOLD}%  "
      f"Accuracy={best_thr_row['Accuracy']:.4f}  "
      f"F1={best_thr_row['F1']:.4f}  "
      f"Coverage={best_thr_row['Coverage (% days)']:.1f}%")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(sweep_df["Threshold (±%)"], sweep_df["Accuracy"],
         color="steelblue", marker="o", linewidth=2, label="Accuracy")
ax1.plot(sweep_df["Threshold (±%)"], sweep_df["F1"],
         color="darkorange", marker="s", linewidth=2, linestyle="--", label="F1")
ax1.axhline(0.55, color="green", linewidth=0.8, linestyle=":", label="55% target")
ax1.axhline(0.65, color="red",   linewidth=0.8, linestyle=":", label="65% target")
ax1.set_xlabel("Neutral threshold (±%)")
ax1.set_ylabel("Score")
ax1.set_ylim(0.40, 0.80)
ax1.legend(loc="upper left")
ax2.bar(sweep_df["Threshold (±%)"], sweep_df["Coverage (% days)"],
        width=0.08, alpha=0.25, color="grey", label="Coverage %")
ax2.set_ylabel("Coverage (% of test days)", color="grey")
ax2.set_ylim(0, 120)
ax2.legend(loc="upper right")
ax1.set_title("Moirai Threshold Sweep — Accuracy / F1 vs Coverage")
plt.tight_layout()
plt.show()

---
## Section 6 — Model Comparison: Moirai vs Baseline

**F1 (weighted)** remains the primary metric.

In [ ]:
y_pred_dummy_series  = pd.Series(y_pred_dummy, index=df_test.index)
y_pred_dummy_aligned = y_pred_dummy_series.loc[common_dates][valid_mask]
baseline_aligned     = evaluate_model(y_test_moirai, y_pred_dummy_aligned, "Baseline (Majority Class)")

results_df = pd.DataFrame([baseline_aligned, moirai_metrics])
print(results_df.to_string(index=False))

metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1"]
x     = np.arange(len(results_df))
width = 0.20

fig, ax = plt.subplots(figsize=(9, 5))
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, results_df[metric], width, label=metric)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(results_df["Model"], rotation=10, ha="right")
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — Moirai Direction Signal vs Baseline")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

best_row = results_df.loc[results_df["F1"].idxmax()]
print(f"\nBest model: {best_row['Model']}   Accuracy={best_row['Accuracy']:.4f}   F1={best_row['F1']:.4f}")

### Comparison Interpretation

| F1 range | Practical meaning |
|---|---|
| < 0.37 (baseline) | Moirai adds no value over always predicting UP |
| 0.37–0.51 | Marginal signal — foundation model captures some structure |
| 0.51–0.58 | Genuine improvement — multivariate context + probabilistic forecasting helps |
| > 0.58 | Strong — approaching the ceiling for 1-day stock direction prediction |

**Comparison with previous iterations:**
- it3 XGBoost: F1 ~0.50–0.52 (technical + macro features, trained classifier)
- it4 ARIMA: F1 ~0.51 (price autocorrelation only)
- it4 ARIMAX: F1 ~0.50 (adding macro signals to ARIMA hurt slightly)
- **it5 Moirai**: zero-shot foundation model with multivariate context

---
## Section 7 — Error Analysis

Moirai prediction errors overlaid on the close price chart — same analysis as it4.

In [ ]:
cm     = confusion_matrix(y_test_moirai, y_pred_moirai)
cm_pct = cm.astype(float) / cm.sum() * 100
annot  = np.array([
    [f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)" for j in range(cm.shape[1])]
    for i in range(cm.shape[0])
])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_pct, annot=annot, fmt="", cmap="Blues",
            xticklabels=["DOWN (0)", "UP (1)"],
            yticklabels=["DOWN (0)", "UP (1)"], ax=ax)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("Confusion Matrix (counts + %) — Moirai Direction Signal")
plt.tight_layout()
plt.show()

eval_index   = y_test_moirai.index
ypred_series = pd.Series(y_pred_moirai.values, index=eval_index)
false_pos    = eval_index[(ypred_series == 1) & (y_test_moirai == 0)]
false_neg    = eval_index[(ypred_series == 0) & (y_test_moirai == 1)]

logger.info("False positives (pred UP, actual DOWN): %d", len(false_pos))
logger.info("False negatives (pred DOWN, actual UP): %d", len(false_neg))

close_test_window = df_raw.loc[df_raw.index >= test_start_date, close_col]

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(close_test_window.index, close_test_window.values,
        color="royalblue", linewidth=1.2, label="MSFT Close")
ax.scatter(false_pos, close_test_window.reindex(false_pos).values,
           color="red", marker="v", s=55, zorder=5,
           label=f"False Positive ({len(false_pos)})")
ax.scatter(false_neg, close_test_window.reindex(false_neg).values,
           color="darkorange", marker="^", s=55, zorder=5,
           label=f"False Negative ({len(false_neg)})")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate(rotation=30)
ax.set_xlabel("Date")
ax.set_ylabel("Close Price (USD)")
ax.set_title("MSFT Prediction Errors Overlay — Moirai Direction Signal")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Predicted return distribution
pred_returns_series = pd.Series(predicted_returns, index=close_test.index)
fig, ax = plt.subplots(figsize=(10, 4))
pred_returns_series.dropna().hist(bins=60, ax=ax, color="steelblue", edgecolor="white")
ax.axvline( NEUTRAL_THRESHOLD, color="red", linestyle="--", label=f"+{NEUTRAL_THRESHOLD}% UP")
ax.axvline(-NEUTRAL_THRESHOLD, color="red", linestyle="--", label=f"-{NEUTRAL_THRESHOLD}% DOWN")
ax.set_xlabel("Moirai predicted 1-day return (%)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Moirai Predicted Returns — Test Window")
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 8 — Business Value Interpretation

### What the model does

1. Moirai is a pre-trained zero-shot foundation model — no fitting on MSFT data
2. Context window: last 128 days of MSFT close + SPY + VIX + gold + oil
3. Returns 100 probabilistic forecast samples for tomorrow's price
4. Median sample → direction signal; 10th/90th percentiles → confidence interval

### Moirai vs Previous Iterations

| Model | Type | Macro signals | Probabilistic | F1 |
|---|---|---|---|---|
| it3 XGBoost | Trained classifier | Yes | No | ~0.52 |
| it4 ARIMA | Statistical time series | No | No | ~0.51 |
| it4 ARIMAX | Statistical + exogenous | Yes | No | ~0.50 |
| **it5 Moirai** | **Foundation model** | **Yes** | **Yes** | **TBD** |

### Limitations

| Limitation | Impact |
|---|---|
| 1-day horizon is inherently noisy | Even perfect information rarely gives >65% accuracy |
| No fine-tuning on MSFT data | Domain-specific patterns not learned |
| Inference time | ~0.5–2 min for 500+ test steps vs seconds for ARIMA |

### Dashboard Integration

The saved `moirai_lucan_it5.pkl` can be used in the dashboard to:
1. Build a 128-day context tensor from the last 128 days of close + covariate data
2. `moirai_model(past_target=..., ...)` → 100 forecast samples
3. Display: median forecast price, predicted return %, uncertainty interval
4. Direction badge: UP (if q10 > current) / DOWN (if q90 < current) / UNCERTAIN
5. Disclaimer: not financial advice